# Human Development Index (HDI) Prediction Model Training

This notebook loads the UNDP Human Development Index dataset, inspects its structure using Pandas, performs exploratory data visualizations, preprocesses indicators for the year 2021, and trains a LinearRegression model.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import pickle
import os

### 1. Load Dataset & Inspect Structure

In [ ]:
# Load the CSV dataset
df = pd.read_csv('../Dataset/HDI.csv')

# Display the shape (rows and columns)
print("Dataset Shape:", df.shape)

# Inspect structural summary
df.info(max_cols=10)

# Display first 5 rows
df.head()

### 2. Data Exploration & Visualizations

Here we perform exploration tasks to understand unique countries and the relationship between the target HDI score and input metrics. To avoid overcrowding in plots, all visualizations use the first 20 rows (`data1`).

In [ ]:
# 2.1 Unique Country Values
unique_countries = df['Country'].unique()
print(f"Total unique countries listed: {len(unique_countries)}")
print(f"Total rows in dataset: {df.shape[0]}")
print("Are there duplicate country entries?", len(unique_countries) != df.shape[0])

# Prepare subset for visualizations using the first 20 rows of cleaned data
columns_layout = [
    'ISO3',
    'Human Development Groups',
    'Country',
    'UNDP Developing Regions',
    'Human Development Index (2021)',
    'Life Expectancy at Birth (2021)',
    'Expected Years of Schooling (2021)',
    'Mean Years of Schooling (2021)',
    'Gross National Income Per Capita (2021)'
]
df_layout = df[columns_layout]
df_clean = df_layout.dropna()
data1 = df_clean.head(20)

# 2.2 Mean Years of Schooling vs HDI (Strip Plot)
plt.figure(figsize=(12, 6))
sns.stripplot(
    data=data1, 
    x='Mean Years of Schooling (2021)', 
    y='Human Development Index (2021)', 
    size=8, 
    color='purple', 
    jitter=0.25
)
plt.title('Mean Years of Schooling vs HDI Score (First 20 Countries)')
plt.xlabel('Mean Years of Schooling (2021)')
plt.ylabel('Human Development Index (2021)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

# 2.3 Life Expectancy vs HDI (Scatter Plot with trend line)
plt.figure(figsize=(12, 6))
sns.scatterplot(
    data=data1, 
    x='Life Expectancy at Birth (2021)', 
    y='Human Development Index (2021)', 
    s=100, 
    color='teal',
    edgecolor='black'
)
sns.regplot(
    data=data1, 
    x='Life Expectancy at Birth (2021)', 
    y='Human Development Index (2021)', 
    scatter=False, 
    color='cyan'
)
plt.title('Life Expectancy vs HDI Score (First 20 Countries)')
plt.xlabel('Life Expectancy at Birth (2021)')
plt.ylabel('Human Development Index (2021)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

# 2.4 Correlation Heatmap
corr_matrix = df_clean.iloc[:, 4:9].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.3f', linewidths=0.5)
plt.title('Correlation Matrix Heatmap of HDI Indicators')
plt.tight_layout()
plt.show()

### 3. Preprocess & Prepare 2021 Metrics

In [ ]:
# Select independent features X and dependent target Y from df_layout
X = df_layout.iloc[:, [5, 6, 7, 8]]
Y = df_layout.iloc[:, 4]

# Count null values in each selected input column of X
print("Null values in X before imputation:")
print(X.isnull().sum())

# Fill null values in X using the column mean
X = X.fillna(X.mean())

# Drop rows where target Y is missing to ensure valid model training
non_null_y_mask = Y.notna()
X = X[non_null_y_mask]
Y = Y[non_null_y_mask]

### 4. Model Training & Evaluation

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# Train LinearRegression
model = LinearRegression()
model.fit(X_train, y_train)

# Generate HDI Predictions
y_pred = model.predict(X_test)
print("Predicted HDI values:")
print(y_pred)

# Calculate R-Squared Value & MSE
print("\nR² Score on Test Set:", r2_score(y_test, y_pred))
print("MSE on Test Set:", mean_squared_error(y_test, y_pred))

# Inspect ground truth (y_test) vs predicted (y_pred)
print("\nActual ground truth HDI scores (y_test) as an array:")
print(y_test.values)
print("\nPredicted HDI scores (y_pred) as an array:")
print(y_pred)

# Test model with a reduced input set (First 5 records)
print("\nValidation on Reduced Input Set (First 5 records of test set):")
comparison_df = pd.DataFrame({
    'Actual (y_test)': y_test.values[:5],
    'Predicted (y_pred)': y_pred[:5],
    'Absolute Error': np.abs(y_test.values[:5] - y_pred[:5])
})
print(comparison_df)

# Train final model on all records
final_model = LinearRegression()
final_model.fit(X, Y)
print("\nFinal model training score (R²):", final_model.score(X, Y))

### 5. Export Pickled Model to Flask Directory

In [ ]:
os.makedirs('../Flask', exist_ok=True)
with open('../Flask/HDI.pkl', 'wb') as f:
    pickle.dump(final_model, f)
print("Model saved to ../Flask/HDI.pkl")